# 10.2 MySQL and PostgreSQL from Python

**Prerequisites:** 10.1 Introduction to SQL  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Popular database engines, and what ODBC is
- Using the MySQL command-line client (DDL / DML / DQL / DCL / TCL)
- Reaching a **server** database from Python through a **PEP 249** driver
- Why `pymysql`, `psycopg` and `sqlite3` all look the same - and where they differ
- 🔴 **Placeholder styles**: `%s` vs `?`, and why you never format SQL yourself
- `AUTO_INCREMENT` vs `SERIAL`/`IDENTITY`, and PostgreSQL's `RETURNING`
- Transactions, autocommit, and which driver commits by default
- The `utf8` vs `utf8mb4` trap in MySQL
- Running a throwaway practice server with Docker

---

### Popular DB:

1) Oracle DB

2) MS SQL Server

3) My SQL Server

4) SqLite Server

5) MongoDB

### ODBC - Open Data Base Connectivity. standard maintained by microsoft.
- Database driver for python programming act as interface between python program and database.
- Packages available in market for database driver are MySQLClient, MySQLdb, MySQLConnector, PyMySQL.
- MySQLClient, MySQLdb, MySQLConnector require additional support packages for python program connectivity with database.
- PyMySQL is python specific complete package for python program connectivity with database.

### DB Courses available:
- DB Admin
- DB Developer - here we learn SQL(Structured Query Language)

> **NOTE:** during installation the default username is `root`, and you choose the password
> yourself.
>
> ### 🔴 A credential was removed from this cell
> The original version of this note contained the author's actual MySQL root password in
> plain text. It has been removed.
>
> **Never put credentials in a notebook, a script, or anything that goes into version
> control.** Notebooks are especially dangerous because they are shared, emailed and
> committed far more casually than source files.
>
> Read them from the environment instead:
>
> ```python
> import os
> password = os.environ["MYSQL_PASSWORD"]     # set outside the code
> ```
>
> Configuration and secrets are covered in **18 Tooling, Packaging and Environments**.

- MySQL command line Client is directly connected database editor comes as default editor with MySQL

---

## How to read this notebook

It has **two halves**.

**The first half** (below) is a transcript of a MySQL command-line session. It is not Python, so it is stored as formatted SQL blocks rather than code cells.

> 🔴 **Correction.** These were originally *code* cells, so Jupyter tried to execute them as Python and every one raised `SyntaxError`. The notebook could not be run at all.

**The second half** connects to real servers from Python and executes. It runs against **MySQL 8** and **PostgreSQL 16** at the same time, because the point is how similar the Python looks and how the SQL differs.

### You do not need a server to run this notebook

Every live cell checks first. With no server reachable it prints the SQL it *would* have run and carries on, so the notebook always executes cleanly top to bottom.

To get real output, start the practice stack:

```bash
docker compose -f "10 Database/docker/docker-compose.yml" up -d
```

It uses **deliberately non-standard ports** (MySQL on 53306, PostgreSQL on 55432) so it cannot collide with a database you actually use, and named volumes so nothing is written into these notes. The passwords in it are throwaway.

### Inside default MySQL editor:

> To resolve the conflict of similar database name, number of database instance can be created inside the account in MySQL.

```sql
# create database db_instance_name;

create database db_table1;

# To enter inside database instance created
# use db_instance_name;

use db_table1;
```

```sql
# Structure of SQL:
#   - DDL(Data ) Command: create, alter etc all database structure related queries
#   - DML(Data ) Command: delete, insert
#   - DQL(Data Query Language) Command: Select
#   - DCL(Data Control Language) Command: grant permission, invoke
#   - TCL(Transaction Control Langugage) Command: commit, rollback, savepoint
```

```sql
# To create a table:
# create table table_name(column1_name column1_datatype(column1_size), column2_name column2_datatype(column2_size) primary key, ...etc);

create table student(name varchar(20), rollno integer(10) primary key, marks integer(10));

# NOTE: for string datatype we have 3 types: varchar(), char(), 
# NOTE: for string datatype we have 2 types: int()/integer(), float()
# NOTE: unique key column concept avoid duplicacy of value. More than one column of this type is allowed in a table.
# NOTE: primary key column concept avoid duplicacy and emptyness of value. Only one column of this type is allowed in a table.
```

```sql
# Hetrogeneous collection of data is called record.
```

```sql
# To see the structure of table created
# desc table_name;

desc student;
```

```sql
# To insert record in default order in which table is created:
# insert into table_name values(value1, value2,...etc);
insert into student values(Shyam,1,500);


# To insert record in manual order:
# insert into table_name(manual column_names order) values(value1, value2,...etc);

insert into student (rollno,name,marks) values(2,Ram,600);
```

```sql
# To retrive information from table and display:
# select column1_name from table_name;

select name from student;

# select column1_name,column2_name from table_name;

select name,marks from student;

# select * from table_name;
select * from student;

# NOTE: * is used to select all columns from table in default order in which table is created.
# NOTE: If you want to control order of selection we can write column name in manual order instead of *.

# To select row 'where' clause is used:
# select * from table_name where column_name opeator condition;
select * from student where marks>500;

# NOTE: Here the rows displayed in default order of insertion.
# NOTE: To select row to be displayed in an order use order by.(default is asc)
# select * from table_name order by column_name in des/asc where column_name opeator with condition;
select * from student order by marks in des where marks>500;
```

```sql
# To update values of existing table structure:
update table_name set column_name = column_name + 10;

# To update values of existing table structure for particular row or condition:
update table_name set column_name = column_name + 10 where column_name = 2 ;
```

```sql
# To delete a particular row:
delete from student where column_name = 2;
```

```sql
# To delete a table:
# using delete we can use where command also
delete from student;
# using truncate we can't use where command as in delete
truncate from student;
# using drop values in table and table structure both get removed
drop table student;
```

```sql
# To save data in database upto today.
commit;
# NOTE: MySQl automatically commit the database
```

```sql
# To exit from database
exit;
```

---

## Reaching MySQL from Python

Everything above was typed at the `mysql>` prompt. To do it from a program you need a
**driver** — a package that speaks MySQL's wire protocol.

| Package | Install | Notes |
|---|---|---|
| **`mysql-connector-python`** | `pip install mysql-connector-python` | Official, from Oracle. Pure Python. |
| **`PyMySQL`** | `pip install PyMySQL` | Pure Python, widely used, no compiler needed |
| **`mysqlclient`** | `pip install mysqlclient` | C extension — fastest, but needs build tools |

All three implement **PEP 249**, the Python Database API. That is the important part: the
*same* `connect` / `cursor` / `execute` / `fetchall` / `commit` shape you will learn in
**10.3** for SQLite works here too. Swapping databases changes the connection line, not the
code around it.

> ⚠️ The original notebook did `from pymysql import *`. Never do that (**7.1**) — it dumps
> every name from the package into your namespace.

In [ ]:
# This cell does not require a MySQL server - it only checks what is available.
import importlib.util

drivers = {
    "mysql.connector": "mysql-connector-python",
    "pymysql": "PyMySQL",
    "MySQLdb": "mysqlclient",
}


def is_installed(module: str) -> bool:
    """find_spec raises if a PARENT package is missing, so guard it."""
    try:
        return importlib.util.find_spec(module) is not None
    except (ImportError, ModuleNotFoundError, ValueError):
        return False


print("MySQL drivers on this machine:")
available = [m for m in drivers if is_installed(m)]
for module, package in drivers.items():
    state = "installed" if module in available else "not installed"
    print(f"  {module:<16} {state:<14} (pip install {package})")

if not available:
    print()
    print("None installed - the code below is shown for reference only.")
    print("10.3 covers the same patterns with sqlite3, which needs no server.")


---

# Part 2: connecting for real

Everything from here executes. The next cell works out what is reachable; every later cell adapts to what it finds.

### Step 1 - what is actually running?

A short TCP probe before touching a driver. Connecting to a dead port with a driver means waiting out its own timeout, which can be many seconds; a socket check with a half-second timeout tells you the same thing immediately.

In [ ]:
import os
import socket


def server_available(host: str, port: int, timeout: float = 0.5) -> bool:
    """Is anything listening? Cheap enough to call before every connect."""
    with socket.socket() as probe:
        probe.settimeout(timeout)
        try:
            probe.connect((host, port))
            return True
        except OSError:
            return False


# Ports come from the environment so you can point these notebooks at your own
# servers without editing any cell. The defaults match the practice stack.
MYSQL_PORT = int(os.environ.get("PYNOTES_MYSQL_PORT", 53306))
PG_PORT = int(os.environ.get("PYNOTES_PG_PORT", 55432))
HOST = os.environ.get("PYNOTES_DB_HOST", "127.0.0.1")

HAVE_MYSQL = server_available(HOST, MYSQL_PORT)
HAVE_PG = server_available(HOST, PG_PORT)

print(f"MySQL      on {HOST}:{MYSQL_PORT}  ->", "reachable" if HAVE_MYSQL else "not running")
print(f"PostgreSQL on {HOST}:{PG_PORT}  ->", "reachable" if HAVE_PG else "not running")

if not (HAVE_MYSQL or HAVE_PG):
    print(
        "\nNeither server is up, so the cells below will print their SQL instead of\n"
        "running it. That is fine - the notebook still works. To get real output:\n"
        '  docker compose -f "10 Database/docker/docker-compose.yml" up -d'
    )

### Step 2 - connecting

🔴 **Credentials never belong in source.** The practice stack uses a throwaway password so this notebook can be self-contained, but it is still read from the environment with a default — which is the pattern you want, not the value.

Notice the two `connect()` calls. Different packages, different databases, **same shape** — that is [PEP 249](https://peps.python.org/pep-0249/), the Python Database API. Learn it once and every SQL database in Python looks familiar.

In [ ]:
mysql_conn = None
pg_conn = None

if HAVE_MYSQL:
    import pymysql

    mysql_conn = pymysql.connect(
        host=HOST,
        port=MYSQL_PORT,
        user=os.environ.get("PYNOTES_MYSQL_USER", "learner"),
        password=os.environ.get("PYNOTES_MYSQL_PASSWORD", "learnpython"),
        database=os.environ.get("PYNOTES_MYSQL_DB", "notes"),
        charset="utf8mb4",          # NOT 'utf8' - see the trap below
        connect_timeout=5,
    )
    print("MySQL      :", mysql_conn.get_server_info())

if HAVE_PG:
    import psycopg

    pg_conn = psycopg.connect(
        host=HOST,
        port=PG_PORT,
        user=os.environ.get("PYNOTES_PG_USER", "learner"),
        password=os.environ.get("PYNOTES_PG_PASSWORD", "learnpython"),
        dbname=os.environ.get("PYNOTES_PG_DB", "notes"),
        connect_timeout=5,
    )
    print("PostgreSQL :", pg_conn.execute("select version()").fetchone()[0][:38])

if mysql_conn is None and pg_conn is None:
    print("no connections - later cells will print their SQL instead")

### A tiny helper so every cell below stays readable

Without it each example needs three lines of `if conn is None` bookkeeping. With it the cells show the SQL and nothing else.

In [ ]:
def run(conn, label, sql, params=(), fetch=False):
    """Execute one statement, or explain why it was skipped.

    Returns the rows when fetch=True, otherwise None.
    """
    headline = " ".join(sql.split())[:66]
    if conn is None:
        print(f"[{label}: no server] {headline}")
        return None

    with conn.cursor() as cur:
        cur.execute(sql, params)
        rows = cur.fetchall() if fetch else None
    print(f"[{label}] {headline}")
    if rows is not None:
        for row in rows:
            print("    ", row)
    return rows


print("helper ready")

### Step 3 - the same table in both dialects

The table is a **job queue**, which is a fair test: it needs a generated key, a constrained status column and a timestamp — the three things dialects disagree about.

| | MySQL | PostgreSQL |
|---|---|---|
| generated key | `INT AUTO_INCREMENT` | `GENERATED ALWAYS AS IDENTITY` |
| text | `VARCHAR(n)` | `VARCHAR(n)` or `TEXT` |
| timestamp default | `TIMESTAMP DEFAULT CURRENT_TIMESTAMP` | `TIMESTAMPTZ DEFAULT now()` |

`SERIAL` still works in PostgreSQL and you will see it everywhere, but `GENERATED ALWAYS AS IDENTITY` is the standard form and has been preferred since PostgreSQL 10.

In [ ]:
# Drop first so the notebook is re-runnable - the whole point of DROP IF EXISTS
run(mysql_conn, "mysql", "DROP TABLE IF EXISTS job")
run(pg_conn, "pg", "DROP TABLE IF EXISTS job")

run(mysql_conn, "mysql", """
    CREATE TABLE job (
        id       INT AUTO_INCREMENT PRIMARY KEY,
        name     VARCHAR(64)  NOT NULL,
        state    VARCHAR(16)  NOT NULL DEFAULT 'queued',
        attempts INT          NOT NULL DEFAULT 0,
        created  TIMESTAMP    NOT NULL DEFAULT CURRENT_TIMESTAMP
    )
""")

run(pg_conn, "pg", """
    CREATE TABLE job (
        id       INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        name     VARCHAR(64)  NOT NULL,
        state    VARCHAR(16)  NOT NULL DEFAULT 'queued',
        attempts INT          NOT NULL DEFAULT 0,
        created  TIMESTAMPTZ  NOT NULL DEFAULT now()
    )
""")

### Step 4 - 🔴 placeholders, and the commit that MySQL needs

**Both** drivers use `%s`. SQLite uses `?`. That difference is a property of the *driver*, not of SQL — PEP 249 calls it `paramstyle`, and you can ask a driver which one it uses.

```
     cur.execute("INSERT INTO job (name) VALUES (%s)", (name,))
                                              ^^^^     ^^^^^^^
                                              |        the value, kept separate
                                              a placeholder, NOT string formatting
```

`%s` here has **nothing to do with Python's `%` operator**. Writing `"... VALUES (%s)" % name` yourself is precisely the SQL-injection hole that **10.3** demonstrates with a working attack.

Note also the trailing comma in `(name,)` — without it that is a plain string, not a one-element tuple.

In [ ]:
if mysql_conn is not None:
    import pymysql
    print("pymysql paramstyle:", pymysql.paramstyle)
if pg_conn is not None:
    import psycopg
    print("psycopg paramstyle:", psycopg.paramstyle)
print("sqlite3 paramstyle:", __import__("sqlite3").paramstyle)
print()

JOBS = [("reindex-search", "queued"), ("purge-cache", "running"), ("send-digest", "queued")]

for name, state in JOBS:
    run(mysql_conn, "mysql", "INSERT INTO job (name, state) VALUES (%s, %s)", (name, state))
    run(pg_conn, "pg", "INSERT INTO job (name, state) VALUES (%s, %s)", (name, state))

# 🔴 PyMySQL does NOT autocommit. Without this the rows vanish when the
# connection closes. psycopg 3 also opens a transaction implicitly.
if mysql_conn is not None:
    mysql_conn.commit()
if pg_conn is not None:
    pg_conn.commit()
if mysql_conn is not None or pg_conn is not None:
    print("\ncommitted")
else:
    print("\nnothing to commit - no server")

### Step 5 - reading it back

Identical SQL, identical Python, two different engines.

In [ ]:
run(mysql_conn, "mysql", "SELECT id, name, state FROM job ORDER BY id", fetch=True)
print()
run(pg_conn, "pg", "SELECT id, name, state FROM job ORDER BY id", fetch=True)
print()
run(mysql_conn, "mysql",
    "SELECT state, COUNT(*) FROM job GROUP BY state ORDER BY state", fetch=True)
print()
run(pg_conn, "pg",
    "SELECT state, COUNT(*) FROM job GROUP BY state ORDER BY state", fetch=True)

### Step 6 - `RETURNING`, which PostgreSQL has and MySQL does not

You have just inserted a row and you need its generated id. The approaches differ:

- **PostgreSQL** — add `RETURNING id` to the `INSERT`. One round trip, and it works for `UPDATE` and `DELETE` too.
- **MySQL** — read `cursor.lastrowid` afterwards.

`lastrowid` is per-cursor and reflects the *last* insert, so with `executemany` it gives you the first id of the batch on some drivers and the last on others. When it matters, insert one row at a time or use `RETURNING` on an engine that has it.

In [ ]:
if pg_conn is not None:
    with pg_conn.cursor() as cur:
        cur.execute(
            "INSERT INTO job (name, state) VALUES (%s, %s) RETURNING id, created",
            ("rebuild-index", "queued"),
        )
        new_id, created = cur.fetchone()
    pg_conn.commit()
    print(f"pg   : RETURNING gave id={new_id} created={created}")
else:
    print("[pg: no server] INSERT ... RETURNING id, created")

if mysql_conn is not None:
    with mysql_conn.cursor() as cur:
        cur.execute("INSERT INTO job (name, state) VALUES (%s, %s)",
                    ("rebuild-index", "queued"))
        print(f"mysql: lastrowid gave id={cur.lastrowid}")
    mysql_conn.commit()
else:
    print("[mysql: no server] INSERT ...; then cursor.lastrowid")

### Step 7 - 🔴 the `utf8` trap in MySQL

MySQL's `utf8` is **not UTF-8.** It is a three-byte-per-character subset that predates the standard, and it cannot store anything outside the Basic Multilingual Plane — emoji, and some CJK characters.

The real UTF-8 in MySQL is called **`utf8mb4`**. Use it for the connection, the database, the table and the column.

> **Version note.** MySQL 8.0 made `utf8mb4` the server default and 8.0.29 began reporting `utf8` as an alias for `utf8mb3`, which is deprecated. On an older server the old meaning may still be in force, so be explicit rather than relying on defaults.

In [ ]:
SAMPLE = "deploy \N{ROCKET} \N{PARTY POPPER}"      # astral-plane characters
print("storing:", SAMPLE)

if mysql_conn is not None:
    with mysql_conn.cursor() as cur:
        cur.execute("INSERT INTO job (name, state) VALUES (%s, %s)", (SAMPLE, "queued"))
    mysql_conn.commit()
    rows = run(mysql_conn, "mysql",
               "SELECT name FROM job WHERE state=%s AND name LIKE %s",
               ("queued", "deploy%"), fetch=True)
    got = rows[0][0] if rows else None
    print("round-tripped intact:", got == SAMPLE)
    print("  ^ because the connection asked for utf8mb4. With charset='utf8'")
    print("    this raises an encoding error or silently mangles the text.")
else:
    print("[mysql: no server] INSERT/SELECT round trip of astral characters")

print()
if pg_conn is not None:
    with pg_conn.cursor() as cur:
        cur.execute("INSERT INTO job (name, state) VALUES (%s, %s) RETURNING name",
                    (SAMPLE, "queued"))
        print("pg round-tripped intact:", cur.fetchone()[0] == SAMPLE)
    pg_conn.commit()
    print("  ^ PostgreSQL has no equivalent trap: UTF8 means UTF-8.")
else:
    print("[pg: no server] same round trip")

### Step 8 - transactions, and who commits for you

This is where the drivers genuinely disagree, and getting it wrong loses data silently.

| | default | to make a change permanent |
|---|---|---|
| `pymysql` | transaction open, **no autocommit** | `conn.commit()` |
| `psycopg` 3 | transaction open on first statement | `conn.commit()`, or use `with conn` |
| `sqlite3` | autocommits DDL, defers DML | `conn.commit()` |

The safe habit is the same everywhere: **be explicit**. Commit on success, roll back on failure, and never assume.

In [ ]:
def attempt_transfer(conn, label, *, fail: bool):
    """Mark two jobs done as one unit. Either both land or neither does."""
    if conn is None:
        print(f"[{label}: no server] UPDATE ... ; UPDATE ... ; commit/rollback")
        return
    try:
        with conn.cursor() as cur:
            cur.execute("UPDATE job SET state='done' WHERE name=%s", ("reindex-search",))
            if fail:
                raise RuntimeError("worker crashed between the two updates")
            cur.execute("UPDATE job SET state='done' WHERE name=%s", ("send-digest",))
        conn.commit()
        print(f"[{label}] committed")
    except RuntimeError as exc:
        conn.rollback()
        print(f"[{label}] rolled back: {exc}")


for conn, label in ((mysql_conn, "mysql"), (pg_conn, "pg")):
    attempt_transfer(conn, label, fail=True)
    run(conn, label, "SELECT COUNT(*) FROM job WHERE state='done'", fetch=True)
    print("    ^ still zero: the first UPDATE was undone by the rollback\n")

for conn, label in ((mysql_conn, "mysql"), (pg_conn, "pg")):
    attempt_transfer(conn, label, fail=False)
    run(conn, label, "SELECT COUNT(*) FROM job WHERE state='done'", fetch=True)
    print()

### Step 9 - tidy up

Drop the table and close the connections. A notebook that leaves state behind gives different answers the second time you run it.

In [ ]:
run(mysql_conn, "mysql", "DROP TABLE IF EXISTS job")
run(pg_conn, "pg", "DROP TABLE IF EXISTS job")

for conn, label in ((mysql_conn, "mysql"), (pg_conn, "pg")):
    if conn is not None:
        conn.commit()
        conn.close()
        print(f"{label}: closed")

print("\nThe containers are still running. To stop them:")
print('  docker compose -f "10 Database/docker/docker-compose.yml" down')
print("To stop them and delete the data as well, add -v")

### Where MySQL differs from SQLite and PostgreSQL

The SQL you have written in **10.1** is mostly portable. These are the edges:

| Feature | MySQL | SQLite | PostgreSQL |
|---|---|---|---|
| Auto-incrementing key | `INT AUTO_INCREMENT` | `INTEGER PRIMARY KEY` | `SERIAL` / `IDENTITY` |
| Show tables | `SHOW TABLES;` | `.tables` or query `sqlite_master` | `\dt` |
| Describe a table | `DESC student;` | `PRAGMA table_info(student);` | `\d student` |
| Create a database | `CREATE DATABASE x;` | connect to a filename | `CREATE DATABASE x;` |
| Select a database | `USE x;` | n/a — the connection is the database |  `\c x` |
| String concat | `CONCAT(a, b)` | `a \|\| b` | `a \|\| b` |
| Limit rows | `LIMIT 10` | `LIMIT 10` | `LIMIT 10` |
| Type strictness | Enforced | **Dynamic** — a TEXT column will accept a number | Enforced |
| Placeholder (Python) | `%s` | `?` | `%s` |

> **SQLite's type system is the surprising one.** Column types are *advisory*: SQLite will
> happily store the string `"abc"` in a column declared `INTEGER`. Since 3.37 you can opt in
> to strictness with `CREATE TABLE ... STRICT`.

---

## Common Mistakes & Pitfalls

1. 🔴 **Putting credentials in a notebook or script.** Read them from the environment. Notebooks get shared and committed far too casually - a real password was found in this very notebook.
2. 🔴 **Building SQL with f-strings or `%`.** `%s` in `execute()` is a *placeholder*, not Python formatting. Formatting it yourself is the injection hole demonstrated in **10.3**.
3. 🔴 **Forgetting `commit()` with PyMySQL.** It does not autocommit. Your inserts run, return no error, and disappear when the connection closes.
4. 🔴 **Using MySQL's `utf8`.** It is a three-byte subset that cannot store emoji. You want `utf8mb4` - on the connection, database, table and column.
5. **Assuming `?` works everywhere.** SQLite uses `?`; MySQL and PostgreSQL use `%s`. Check `driver.paramstyle`.
6. **Storing SQL in a Python code cell.** It is not Python and raises `SyntaxError`. Use a markdown cell with a ```sql fence, or execute it through a driver.
7. **Relying on `lastrowid` after `executemany`.** Its meaning varies by driver. Use `RETURNING` where the engine supports it.

## Best Practices

- Read credentials from environment variables or a secrets manager - never from code.
- Always pass parameters as the second argument to `execute()`.
- Set `charset='utf8mb4'` on every MySQL connection.
- Commit explicitly on success and roll back on failure; do not rely on driver defaults.
- Close connections with `with` or `try/finally`.
- Give each application its own database user with the narrowest privileges that work.
- Use `DROP TABLE IF EXISTS` at the top of throwaway scripts so they are re-runnable.
- Probe the port before connecting when a server may be absent - it turns a multi-second driver timeout into a half-second check.

## Practice Exercises

Try these before moving on.

1. Start the practice stack and re-run Part 2. Compare the `created` column between the two engines - why does PostgreSQL show a timezone and MySQL not?
2. Rewrite the `job` table for SQLite and run it in **10.1**. Which three lines change?
3. Add a `CHECK` constraint restricting `state` to queued/running/done, then try to insert an invalid one. Do both engines reject it?
4. Write `get_connection()` that reads host, port, user and password from the environment and raises a clear error naming the missing variable.
5. 🔴 Deliberately rewrite one `execute()` to build its SQL with an f-string, then insert a job named `x'); DROP TABLE job; --`. Explain what saved you (see **10.3**).
6. Insert a row with PyMySQL and *do not* commit. Close the connection, reconnect, and confirm the row is gone.